# Task 3 — Semantic Filter and Exception Balance E9 Experiments

This Colab runner trains only the two predeclared E9 children. Usage runs first from the exact E2 parents. Gender then uses the exact E6 parents after the deterministic 305-row filter contract is reproduced. No human-rating gate is used. It never trains E1–E8.


## 1. Safe Colab setup

Run on a GPU runtime. The repository update stops on local changes. Training data is extracted into Colab-local storage; evidence and the registry stay in Drive.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=cwd, check=True)


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("This runner must run in Google Colab.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError(
            "The Colab clone has local changes. Save them first; no switch or merge was attempted."
        )
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

print(f"Repository ready: {REPO_DIR} ({BRANCH})")


In [ ]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}
image_dirs = (
    teacher_dir / "train/images_train",
    teacher_dir / "test/images_test",
)

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [
        name for name in names if Path(name).is_absolute() or ".." in Path(name).parts
    ]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/")
        and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in the expected folder.")
    current_images = sum(
        path.suffix.lower() in image_suffixes
        for image_dir in image_dirs
        for path in image_dir.glob("*")
    )
    needs_extract = current_images != expected_images or not all(
        path.is_file() for path in required_files
    )
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(
    path.suffix.lower() in image_suffixes
    for image_dir in image_dirs
    for path in image_dir.glob("*")
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found "
        f"{actual_images:,}; missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
DRIVE_TASK_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print("Output folders are ready.")


## 2. Exact parents and zero-step preflight

The IDs below are the validated completed parent folds. The resolver must return the same tuple. Preflight builds the child shape and contract but creates no optimizer step.


In [ ]:
from fashion.train.task3_experiments import (
    audit_completed_registry_rows,
    check_task3_child_setup,
    latest_completed_gender_e6_parent_run_ids,
    latest_completed_usage_e2_parent_run_ids,
    run_task3_child_cv,
)
from fashion.train.task3_e9 import write_task3_e9_prerun_evidence

GENDER_E6_PARENT_RUN_IDS = ('t3_gender_e6_gem_p3_gender_smallcnngem3_f0_s2753_a8c09286451b_20260831T090059Z0bab1f', 't3_gender_e6_gem_p3_gender_smallcnngem3_f1_s2753_a8c09286451b_20260831T090940Z6e10b5', 't3_gender_e6_gem_p3_gender_smallcnngem3_f2_s2753_a8c09286451b_20260831T091823Zabb677', 't3_gender_e6_gem_p3_gender_smallcnngem3_f3_s2753_a8c09286451b_20260831T092710Zee7c6a', 't3_gender_e6_gem_p3_gender_smallcnngem3_f4_s2753_a8c09286451b_20260831T093553Z63b5fd')
USAGE_E2_PARENT_RUN_IDS = ('t3_usage_e2_class_balanced_ce_usage_smallcnn_f0_s2753_5461e048c3b3_20260830T115815Z356f6d', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f1_s2753_5461e048c3b3_20260830T120645Z6d08cd', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f2_s2753_5461e048c3b3_20260830T121514Zd58aa0', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f3_s2753_5461e048c3b3_20260830T122347Z2cb06e', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f4_s2753_5461e048c3b3_20260830T123218Z94db47')

resolved_gender = latest_completed_gender_e6_parent_run_ids(output_root=DRIVE_TASK_DIR)
resolved_usage = latest_completed_usage_e2_parent_run_ids(output_root=DRIVE_TASK_DIR)
if resolved_gender != GENDER_E6_PARENT_RUN_IDS:
    raise RuntimeError("Resolved Gender E6 parents differ from the frozen tuple.")
if resolved_usage != USAGE_E2_PARENT_RUN_IDS:
    raise RuntimeError("Resolved Usage E2 parents differ from the frozen tuple.")

usage_check = check_task3_child_setup(
    "usage_exception_balance",
    parent_run_ids=USAGE_E2_PARENT_RUN_IDS,
    root=REPO_DIR,
    device_name="cuda",
)
gender_check = check_task3_child_setup(
    "gender_semantic_filter",
    parent_run_ids=GENDER_E6_PARENT_RUN_IDS,
    root=REPO_DIR,
    device_name="cuda",
)
if usage_check["optimizer_steps"] != 0 or gender_check["optimizer_steps"] != 0:
    raise RuntimeError("Preflight unexpectedly reported an optimizer step.")
print("Both E9 contracts passed zero-step preflight.")
print("Gender deterministic audit required:", gender_check["gender_deterministic_audit_required"])
print("Gender human gate required:", gender_check["gender_human_rating_gate_required"])


In [ ]:
USAGE_E2_OOF = (
    DRIVE_TASK_DIR
    / "experiments/t3_usage_e2_class_balanced_ce/usage/aggregate/oof_predictions.csv"
)
missing_audit_inputs = [
    str(path) for path in (USAGE_E2_OOF,)
    if not path.is_file()
]
if missing_audit_inputs:
    raise FileNotFoundError(f"Missing saved audit inputs: {missing_audit_inputs}")

e9_prerun = write_task3_e9_prerun_evidence(
    splits_path=REPO_DIR / "data/processed/splits.csv",
    usage_prediction_path=USAGE_E2_OOF,
    output_dir=DRIVE_TASK_DIR / "e9_prerun",
)
if e9_prerun["optimizer_steps"] != 0:
    raise RuntimeError("The audit pack unexpectedly reported an optimizer step.")
gender_contract = e9_prerun["gender_deterministic_contract"]
if not gender_contract["verified"] or not all(gender_contract["checks"].values()):
    raise RuntimeError("The deterministic Gender contract did not verify.")
if gender_contract["human_rating_gate_required"]:
    raise RuntimeError("E9G must not depend on a human-rating gate.")
print("Gender conflict rows:", gender_contract["conflict_rows"])
print("Gender fold-training removals:", gender_contract["training_removals"])
print("Gender excluded-ID hash:", gender_contract["conflict_ids_hash"])
print("Deterministic E9 evidence ready in Drive; optimizer steps: 0")


## 3. Usage E9U — run first

This first training call keeps every row and changes only fold-training usual/exception weights on top of the exact E2 class weights. ArticleType remains training metadata only.


In [ ]:
usage_e9 = run_task3_child_cv(
    "usage_exception_balance",
    parent_run_ids=USAGE_E2_PARENT_RUN_IDS,
    output_root=DRIVE_TASK_DIR,
    folds=range(5),
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[LOCAL_REGISTRY],
    root=REPO_DIR,
    device_name="cuda",
)
usage_registry_audit = audit_completed_registry_rows(
    DRIVE_REGISTRY, usage_e9["fold_run_ids"]
)
print("Usage E9 complete:", usage_e9["metrics_path"])
print(usage_registry_audit)


## 4. Deterministic Gender contract

The next cell rechecks the automated evidence created before training. It needs the exact rule version, 305 conflict IDs, frozen ID and distribution hashes, and fold-removal counts.


In [ ]:
expected_gender_removals = {"0": 242, "1": 224, "2": 259, "3": 263, "4": 232}
if gender_contract["conflict_rows"] != 305:
    raise RuntimeError("Gender conflict count changed.")
if gender_contract["training_removals"] != expected_gender_removals:
    raise RuntimeError("Gender fold-removal counts changed.")
if gender_contract["human_rating_gate_required"]:
    raise RuntimeError("A removed human-rating gate was reintroduced.")
print("Deterministic Gender contract verified; no formal ratings are required.")


## 5. Gender E9G — run after Usage

This call changes only fold-training row selection from the exact E6 GeM parent. It excludes the deterministic conflicts from each training complement, keeps validation untouched, and stores the exact IDs, rule hash, and verification fields with each fold.


In [ ]:
gender_e9 = run_task3_child_cv(
    "gender_semantic_filter",
    parent_run_ids=GENDER_E6_PARENT_RUN_IDS,
    output_root=DRIVE_TASK_DIR,
    folds=range(5),
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[LOCAL_REGISTRY],
    root=REPO_DIR,
    device_name="cuda",
)
gender_registry_audit = audit_completed_registry_rows(
    DRIVE_REGISTRY, gender_e9["fold_run_ids"]
)
print("Gender E9 complete:", gender_e9["metrics_path"])
print(gender_registry_audit)


## 6. Stop and return to the main notebook

Do not run another seed or stack another change here. Return the saved E9 aggregates to Notebook 04, apply the frozen gates, and record accept/reject before any further experiment.
